In [2]:
import cv2
import json

# === Set image path ===
IMAGE_PATH = "plain.jpg"  # Change this to your actual image file
OUTPUT_FILE = "lshape_image_plain.json"

# Store up to 3 clicked points
points = []

# Mouse callback to collect L-shape points
def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        points.append((x, y))

# Load the image
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()

    # Draw selected points
    for point in points:
        cv2.circle(display_frame, point, 5, (0, 0, 255), -1)

    # Draw L-shape lines
    if len(points) >= 2:
        cv2.line(display_frame, points[0], points[1], (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, points[1], points[2], (0, 255, 0), 2)

    # Instructions
    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    cv2.imshow("Define L-Shape ROI", display_frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape to {OUTPUT_FILE}")
    elif key == ord('r'):
        points = []
    elif key == ord('q'):
        break

cv2.destroyAllWindows()
cv2.waitKey(1)
cv2.waitKey(1)

[✅] Saved L-shape to lshape_image_plain.json


-1

In [4]:
import cv2
import json

# === Set image path ===
IMAGE_PATH = "plain.jpeg"  # Change this to your actual image file
OUTPUT_FILE = "lshape_image.json"

# Store up to 3 clicked points
points = []

# Mouse callback to collect L-shape points
def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        points.append((x, y))

# Load the image
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

cv2.namedWindow("Define L-Shape ROI")
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()

    # Draw selected points
    for point in points:
        cv2.circle(display_frame, point, 5, (0, 0, 255), -1)

    # Draw L-shape lines
    if len(points) >= 2:
        cv2.line(display_frame, points[0], points[1], (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, points[1], points[2], (0, 255, 0), 2)

    # Instructions
    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    cv2.imshow("Define L-Shape ROI", display_frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape to {OUTPUT_FILE}")
    elif key == ord('r'):
        points = []
    elif key == ord('q'):
        break

cv2.destroyAllWindows()
cv2.waitKey(1)
cv2.waitKey(1)

[✅] Saved L-shape to lshape_image.json


-1

In [ ]:
import cv2
import json
import numpy as np
from ultralytics import YOLO  # pip install ultralytics

# === Load L-shape points from JSON ===
with open("lshape_image.json", "r") as f:
    l_points = json.load(f)

A, B, C = [tuple(map(int, pt)) for pt in l_points]  # Ensure points are tuples of int

# === Create L-shape lines ===
line1 = (np.array(A), np.array(B))
line2 = (np.array(B), np.array(C))

# === Load YOLOv8 model ===
model_path = "yolo11n.pt"  # Replace with your trained model path
model = YOLO(model_path)
print(f"✅ YOLO model loaded from: {model_path}")

# === Load test image ===
image_path = "test_image.jpeg"  # Change to your actual test image
frame = cv2.imread(image_path)
if frame is None:
    print(f"❌ Failed to load image: {image_path}")
    exit()

# === Run YOLOv8 detection ===
results = model(frame)

# === Draw YOLO detections ===
for box in results[0].boxes:
    cls_id = int(box.cls[0])
    label = model.names[cls_id]
    conf = box.conf[0]

    if "truck" in label.lower():  # You can change to 'person' or others
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{label} {conf:.2f}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

# === Draw L-shape on image ===
cv2.line(frame, A, B, (0, 255, 255), 2)
cv2.line(frame, B, C, (0, 255, 255), 2)

cv2.circle(frame, A, 5, (0, 0, 255), -1)
cv2.circle(frame, B, 5, (0, 255, 0), -1)
cv2.circle(frame, C, 5, (255, 0, 0), -1)

cv2.putText(frame, "A", A, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
cv2.putText(frame, "B", B, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
cv2.putText(frame, "C", C, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

# === Show image ===
cv2.namedWindow("Test Detection with L-Shape", cv2.WINDOW_NORMAL)
#cv2.resizeWindow("Test Detection with L-Shape", 1000, 800)
cv2.imshow("Test Detection with L-Shape", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

✅ YOLO model loaded from: yolo11n.pt

0: 640x480 1 truck, 2 chairs, 1 couch, 47.2ms
Speed: 1.6ms preprocess, 47.2ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 480)


: 

In [ ]:
# 3 jan

In [ ]:
import cv2
import json
import numpy as np

IMAGE_PATH = "test_images/images/plain.jpg"
OUTPUT_FILE = "plain_lshape_image.json"

points = []
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

height, width = frame.shape[:2]
overlay_drawn = None

def to_list(pt):
    return [int(pt[0]), int(pt[1])]

def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        math_y = height - y
        points.append((x, math_y))

def draw_math_grid(img, spacing=50, color=(200, 200, 200), thickness=1):
    for y in range(0, height, spacing):
        y_cv = height - y
        cv2.line(img, (0, y_cv), (width, y_cv), color, thickness)
        cv2.putText(img, f"{y}", (5, y_cv - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), color, thickness)
        cv2.putText(img, f"{x}", (x + 2, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100, 255, 255), 1)

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

def draw_point_label(img, pt, label):
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 0), -1)
        cv2.putText(img, label, (cx + 8, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()
    draw_math_grid(display_frame, spacing=50)

    if overlay_drawn is not None:
        display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

    for (x, my) in points:
        cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

    if len(points) >= 2:
        cv2.line(display_frame, (points[0][0], height - points[0][1]),
                 (points[1][0], height - points[1][1]), (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, (points[1][0], height - points[1][1]),
                 (points[2][0], height - points[2][1]), (0, 255, 0), 2)

    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(display_frame, "Math-style Y axis ↑ (grid overlay)",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 255), 1)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        overlay_drawn = np.zeros_like(frame)

        A, B, C = points
        m_ab, c_ab, eq_ab = line_equation(A, B)
        m_bc, c_bc, eq_bc = line_equation(B, C)

        Q = P = O = None
        if m_ab is not None:
            Q = (-c_ab / m_ab, 0)
            P = (0, c_ab)
            O = (0, 0)
            triangle_pts = np.array([
                (int(Q[0]), height - int(Q[1])),
                (int(P[0]), height - int(P[1])),
                (int(O[0]), height - int(O[1]))
            ], dtype=np.int32)
            cv2.fillPoly(overlay_drawn, [triangle_pts], (255, 0, 255))
            for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
                draw_point_label(overlay_drawn, pt, label)

        if m_bc is not None:
            S = (0, c_bc)
            R = (width, m_bc * width + c_bc)
        else:
            S = (B[0], 0)
            R = (B[0], height)

        bottom_left = (0, 0)
        bottom_right = (width, 0)

        extended_bc_area = np.array([
            (int(S[0]), height - int(S[1])),
            (int(R[0]), height - int(R[1])),
            (bottom_right[0], height - bottom_right[1]),
            (bottom_left[0], height - bottom_left[1])
        ], dtype=np.int32)
        cv2.fillPoly(overlay_drawn, [extended_bc_area], (0, 200, 200))

        for pt, label in zip([A, B, C], ['A', 'B', 'C']):
            draw_point_label(overlay_drawn, pt, label)
        for pt, label in zip([S, R, bottom_right, bottom_left], ['S', 'R', 'BR', 'BL']):
            draw_point_label(overlay_drawn, pt, label)

        # Save all data (convert all to lists)
        save_data = {
            "A": to_list(A),
            "B": to_list(B),
            "C": to_list(C),
            "line_AB": eq_ab,
            "line_BC": eq_bc,
            "Q": to_list(Q) if Q else None,
            "P": to_list(P) if P else None,
            "O": to_list(O) if O else None,
            "S": to_list(S),
            "R": to_list(R),
            "BOTTOM_RIGHT": to_list(bottom_right),
            "BOTTOM_LEFT": to_list(bottom_left)
        }

        with open(OUTPUT_FILE, 'w') as f:
            json.dump(save_data, f, indent=2)

        print(f"[✅] Extended L-shape data saved to: {OUTPUT_FILE}")

    elif key == ord('r'):
        points = []
        overlay_drawn = None
    elif key == ord('q'):
        break

    cv2.imshow("Define L-Shape ROI", display_frame)

cv2.destroyAllWindows()


[✅] Extended L-shape data saved to: plain_lshape_image.json


: 